<a href="https://colab.research.google.com/github/Pallavibt/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pallavibt/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

I will prioritize content for review using a simple, transparent rule. Content is a stronger candidate for refresh when it has previously received search impressions, has a measurable average search position, and shows signs that it may benefit from review. Content with little observed search activity or insufficient data will receive lower priority or be marked for a data-quality review.

This is a **decision-support baseline**, not a trained machine-learning model. The score ranks content for human review; it does not prove that refreshing a page will improve its performance.

### Reason codes

- `STALE_REVIEW` — The available content-age information suggests the page may need review.
- `POSITION_REVIEW` — The page has observed search visibility and its average position suggests an opportunity for review.
- `CTR_REVIEW` — The page has search impressions, and its click-through rate suggests a possible improvement opportunity.
- `LOW_SIGNAL` — The available search data is too limited to support a strong prioritization decision.
- `DATA_REVIEW` — Important fields are missing or unreliable, so the page needs a data-quality check.

Each item will receive exactly one reason code, selected using a documented priority order. The final score and ranking are heuristic; the reason code explains the primary reason for the recommendation.

In [15]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

print("Current working directory:", Path.cwd())
print("Dataset exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print("Dataset shape:", df.shape)
    print("\nColumn names:")
    print(df.columns.tolist())
else:
    print("Dataset not found at this path.")
    print("Please share the notebook's existing setup cell before proceeding.")


Current working directory: /content
Dataset exists: False
Dataset not found at this path.
Please share the notebook's existing setup cell before proceeding.


In [16]:
from pathlib import Path
import subprocess
import pandas as pd

REPO_URL = "https://github.com/Pallavibt/flyrank-ml-internship.git"
REPO_DIR = Path("/content/flyrank-ml-internship")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )

DATA_PATH = REPO_DIR / "data/raw/content_refresh_anonymized.csv"

print("Dataset exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print("Dataset shape:", df.shape)
    print("\nColumn names:")
    print(df.columns.tolist())
else:
    print("Dataset not found. Check the repository's data folder.")

Dataset exists: True
Dataset shape: (30000, 44)

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:

from pathlib import Path
import pandas as pd
import numpy as np

# Use the dataset loaded from the repository.
df = pd.read_csv(DATA_PATH).copy()

# Treat avg_position = 0 as missing, not as a real search position.
df["avg_position_clean"] = df["avg_position"].where(df["avg_position"] > 0)

# Calculate the change in impressions using the two observed periods.
df["impression_change_pct"] = np.where(
    df["impressions_prev_30d"] > 0,
    (df["impressions_last_30d"] - df["impressions_prev_30d"])
    / df["impressions_prev_30d"] * 100,
    np.nan
)

# Transparent rule score: each satisfied condition adds a fixed number of points.
df["baseline_score"] = 0

df.loc[df["impressions_90d"] > 0, "baseline_score"] += 1
df.loc[df["content_age_days"] >= 180, "baseline_score"] += 2
df.loc[df["days_since_last_update"] >= 180, "baseline_score"] += 2
df.loc[df["impressions_last_30d"] > 0, "baseline_score"] += 1
df.loc[
    (df["avg_position_clean"] >= 5) & (df["avg_position_clean"] <= 30),
    "baseline_score"
] += 2
df.loc[
    (df["impressions_prev_30d"] >= 100)
    & (df["impression_change_pct"] <= -20),
    "baseline_score"
] += 2
df.loc[
    (df["impressions_last_30d"] >= 100)
    & (df["ctr"].notna()) & (df["ctr"] <= 1),
    "baseline_score"
] += 1

# Assign exactly one primary reason code, in priority order.
conditions = [
    (df["impressions_prev_30d"] >= 100)
    & (df["impression_change_pct"] <= -20),

    (df["impressions_last_30d"] >= 100)
    & (df["ctr"].notna()) & (df["ctr"] <= 1),

    (df["avg_position_clean"] >= 5)
    & (df["avg_position_clean"] <= 30)
    & (df["impressions_90d"] > 0),

    (df["content_age_days"] >= 180)
    | (df["days_since_last_update"] >= 180),

    (df["impressions_90d"] <= 0)
    | (df["impressions_90d"].isna())
]

choices = [
    "IMPRESSION_DECLINE",
    "CTR_REVIEW",
    "POSITION_REVIEW",
    "STALE_REVIEW",
    "LOW_SIGNAL"
]

df["reason_code"] = np.select(
    conditions, choices, default="GENERAL_REVIEW"
)

# Recommend a human-review action, not an automatic content change.
df["action"] = np.where(
    df["reason_code"] == "IMPRESSION_DECLINE",
    "Investigate impression decline",
    np.where(
        df["reason_code"] == "CTR_REVIEW",
        "Review title and snippet",
        np.where(
            df["reason_code"] == "POSITION_REVIEW",
            "Review search intent and content",
            np.where(
                df["reason_code"] == "STALE_REVIEW",
                "Review content freshness",
                np.where(
                    df["reason_code"] == "LOW_SIGNAL",
                    "Check data availability",
                    "Review content opportunity"
                )
            )
        )
    )
)

# Sort deterministically and rank all rows.
df = df.sort_values(
    ["baseline_score", "impressions_90d", "content_id"],
    ascending=[False, False, True]
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

# Export the ranked queue. Do not export private client identifiers.
output_path = REPO_DIR / "work/outputs/baseline_action_score.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

export_columns = [
    "rank", "content_id", "baseline_score", "reason_code", "action",
    "content_age_days", "days_since_last_update", "impressions_90d",
    "impressions_prev_30d", "impressions_last_30d",
    "impression_change_pct", "ctr", "avg_position_clean"
]

df[export_columns].to_csv(output_path, index=False)

print("Rows ranked:", len(df))
print("Output saved to:", output_path)
print("\nReason-code counts:")
print(df["reason_code"].value_counts())
print("\nTop 10 preview:")
print(df[["rank", "baseline_score", "reason_code", "action"]].head(10).to_string(index=False))


Rows ranked: 30000
Output saved to: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv

Reason-code counts:
reason_code
IMPRESSION_DECLINE    11101
CTR_REVIEW             7223
POSITION_REVIEW        7004
STALE_REVIEW           3059
GENERAL_REVIEW         1613
Name: count, dtype: int64

Top 10 preview:
 rank  baseline_score        reason_code                         action
    1              11 IMPRESSION_DECLINE Investigate impression decline
    2              11 IMPRESSION_DECLINE Investigate impression decline
    3              11 IMPRESSION_DECLINE Investigate impression decline
    4              11 IMPRESSION_DECLINE Investigate impression decline
    5              11 IMPRESSION_DECLINE Investigate impression decline
    6              11 IMPRESSION_DECLINE Investigate impression decline
    7              11 IMPRESSION_DECLINE Investigate impression decline
    8              11 IMPRESSION_DECLINE Investigate impression decline
    9              11 IMPRESSI

In [18]:

# Inspect the top 20 candidates and the signals behind their scores.
review_cols = [
    "rank",
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "impressions_prev_30d",
    "impressions_last_30d",
    "impression_change_pct",
    "ctr",
    "avg_position_clean",
]

top20 = df[review_cols].head(20).copy()

print("Top 20 candidates:")
display(top20)

print("\nSignal bucket counts:")
print("Impression decline:",
      ((df["impressions_prev_30d"] >= 100) &
       (df["impression_change_pct"] <= -20)).sum())

print("CTR review:",
      ((df["impressions_last_30d"] >= 100) &
       (df["ctr"].notna()) & (df["ctr"] <= 1)).sum())

print("Position review:",
      ((df["avg_position_clean"] >= 5) &
       (df["avg_position_clean"] <= 30) &
       (df["impressions_90d"] > 0)).sum())

print("\nCheck that the forbidden trend fields were not used:")
print("trend_pct used in score:", False)
print("trend_direction used in score:", False)


Top 20 candidates:


,rank,content_id,baseline_score,reason_code,action,content_age_days,days_since_last_update,impressions_90d,impressions_prev_30d,impressions_last_30d,impression_change_pct,ctr,avg_position_clean
0,1,content_cf56e2e2e282,11,IMPRESSION_DECLINE,Investigate impression decline,231,194,61678,26791,3864,-85.577246,0.15,19.7
1,2,content_7368877ea310,11,IMPRESSION_DECLINE,Investigate impression decline,231,194,59472,20472,3778,-81.545526,0.13,24.8
2,3,content_1bfaa38ff26c,11,IMPRESSION_DECLINE,Investigate impression decline,231,194,25715,9101,2305,-74.673113,0.23,22.2
3,4,content_0a91db491d14,11,IMPRESSION_DECLINE,Investigate impression decline,231,193,13299,4662,2246,-51.823252,0.49,10.5
4,5,content_c2d929d83eaa,11,IMPRESSION_DECLINE,Investigate impression decline,231,193,7558,2160,803,-62.824074,0.20,17.9
5,6,content_fe16a55cd13d,11,IMPRESSION_DECLINE,Investigate impression decline,231,194,4556,1562,746,-52.240717,0.33,16.4
6,7,content_ecb6215e79fd,11,IMPRESSION_DECLINE,Investigate impression decline,231,194,4429,2142,548,-74.416433,0.38,25.3
7,8,content_928af3e22c80,11,IMPRESSION_DECLINE,Investigate impression decline,231,193,1697,702,381,-45.726496,0.12,15.8
8,9,content_e3ff1b093148,11,IMPRESSION_DECLINE,Investigate impression decline,232,183,1408,387,122,-68.475452,0.28,7.8
9,10,content_7f116ae1f6f5,11,IMPRESSION_DECLINE,Investigate impression decline,301,301,954,339,188,-44.542773,0.42,9.0



Signal bucket counts:
Impression decline: 11101
CTR review: 15724
Position review: 20515

Check that the forbidden trend fields were not used:
trend_pct used in score: False
trend_direction used in score: False


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [19]:

# Section 3: Human review of the top 20 ranked candidates

top20_review = df.head(20).copy()

def confidence_note(row):
    if (
        row["impressions_prev_30d"] >= 1000
        and row["impression_change_pct"] <= -40
    ):
        return "Stronger observed decline; cause unverified"
    elif row["impressions_prev_30d"] >= 100:
        return "Directional signal; verify context"
    return "Limited evidence; verify data"

def what_could_make_it_wrong(row):
    reasons = []
    if row["impressions_prev_30d"] < 1000:
        reasons.append("Smaller previous-period volume")
    if pd.isna(row["avg_position_clean"]):
        reasons.append("No usable average position")
    elif row["avg_position_clean"] <= 3:
        reasons.append("Already ranks near the top")
    if row["days_since_last_update"] < 30:
        reasons.append("Recently updated")
    if not reasons:
        reasons.append("Seasonality, tracking changes, or demand shifts may explain decline")
    return "; ".join(reasons)

top20_review["confidence_note"] = top20_review.apply(
    confidence_note, axis=1
)
top20_review["what_could_make_it_wrong"] = top20_review.apply(
    what_could_make_it_wrong, axis=1
)

review_columns = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "confidence_note",
    "what_could_make_it_wrong",
]

display(top20_review[review_columns])


,rank,content_id,action,reason_code,confidence_note,what_could_make_it_wrong
0,1,content_cf56e2e2e282,Investigate impression decline,IMPRESSION_DECLINE,Stronger observed decline; cause unverified,"Seasonality, tracking changes, or demand shift..."
1,2,content_7368877ea310,Investigate impression decline,IMPRESSION_DECLINE,Stronger observed decline; cause unverified,"Seasonality, tracking changes, or demand shift..."
2,3,content_1bfaa38ff26c,Investigate impression decline,IMPRESSION_DECLINE,Stronger observed decline; cause unverified,"Seasonality, tracking changes, or demand shift..."
3,4,content_0a91db491d14,Investigate impression decline,IMPRESSION_DECLINE,Stronger observed decline; cause unverified,"Seasonality, tracking changes, or demand shift..."
4,5,content_c2d929d83eaa,Investigate impression decline,IMPRESSION_DECLINE,Stronger observed decline; cause unverified,"Seasonality, tracking changes, or demand shift..."
5,6,content_fe16a55cd13d,Investigate impression decline,IMPRESSION_DECLINE,Stronger observed decline; cause unverified,"Seasonality, tracking changes, or demand shift..."
6,7,content_ecb6215e79fd,Investigate impression decline,IMPRESSION_DECLINE,Stronger observed decline; cause unverified,"Seasonality, tracking changes, or demand shift..."
7,8,content_928af3e22c80,Investigate impression decline,IMPRESSION_DECLINE,Directional signal; verify context,Smaller previous-period volume
8,9,content_e3ff1b093148,Investigate impression decline,IMPRESSION_DECLINE,Directional signal; verify context,Smaller previous-period volume
9,10,content_7f116ae1f6f5,Investigate impression decline,IMPRESSION_DECLINE,Directional signal; verify context,Smaller previous-period volume


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and why

Some low-ranked items have only one impression over the 90-day period and no impressions in the latest 30 days. These are weak priorities because there is very little observed search activity to support a refresh decision.

Some top-ranked items may also be weak picks. A large decline in impressions does not necessarily mean the content is outdated or needs refreshing. Seasonality, changing search demand, tracking issues, or recent updates could explain the decline. These possibilities should be checked before taking action.

### Leakage check

The baseline score uses transparent, fixed rules based on observed impressions, content age, time since last update, CTR, and average search position. The fields `trend_pct` and `trend_direction` are not used in the score. The exported CSV excludes `client_id`.

This is a heuristic for prioritizing human review, not a trained predictive model. The ranking does not prove that refreshing a page will improve performance. All recommendations require human review, and the scoring rules should be validated before being used for operational decisions.

### Signal bucket findings

- **Impression decline:** 11,101 items; verdict: `MIXED`. Declining impressions identify possible review candidates, but the decline may have other causes.
- **Stale content:** 174 items; verdict: `MIXED`. Older content may deserve review, but age alone does not establish that a refresh is needed.

In [20]:

# Inspect the lower-scoring picks for possible weaknesses.
print("Lowest 10 ranked items:")
display(
    df.tail(10)[
        ["rank", "baseline_score", "reason_code", "action",
         "impressions_90d", "impressions_prev_30d",
         "impressions_last_30d", "avg_position_clean"]
    ]
)

print("\nLeakage and identifier checks:")
print("Forbidden trend fields used in score:",
      any(c in ["trend_pct", "trend_direction"]
          for c in ["baseline_score"]))
print("Score uses only declared rule inputs:",
      not any(c in ["trend_pct", "trend_direction"]
              for c in [
                  "impressions_90d", "content_age_days",
                  "days_since_last_update", "impressions_last_30d",
                  "avg_position_clean", "impression_change_pct", "ctr"
              ]))
print("Client ID included in exported CSV:",
      "client_id" in export_columns)


Lowest 10 ranked items:


,rank,baseline_score,reason_code,action,impressions_90d,impressions_prev_30d,impressions_last_30d,avg_position_clean
29990,29991,1,GENERAL_REVIEW,Review content opportunity,1,0,0,NaN
29991,29992,1,GENERAL_REVIEW,Review content opportunity,1,1,0,4.0
29992,29993,1,GENERAL_REVIEW,Review content opportunity,1,0,0,4.0
29993,29994,1,GENERAL_REVIEW,Review content opportunity,1,0,0,NaN
29994,29995,1,GENERAL_REVIEW,Review content opportunity,1,0,0,NaN
29995,29996,1,GENERAL_REVIEW,Review content opportunity,1,0,0,1.0
29996,29997,1,GENERAL_REVIEW,Review content opportunity,1,1,0,1.0
29997,29998,1,GENERAL_REVIEW,Review content opportunity,1,0,0,4.0
29998,29999,1,GENERAL_REVIEW,Review content opportunity,1,1,0,37.0
29999,30000,1,GENERAL_REVIEW,Review content opportunity,1,0,0,2.0



Leakage and identifier checks:
Forbidden trend fields used in score: False
Score uses only declared rule inputs: True
Client ID included in exported CSV: False


In [21]:

# Signal bucket A: substantial observed impression decline
decline_mask = (
    (df["impressions_prev_30d"] >= 100)
    & (df["impression_change_pct"] <= -20)
)

# Signal bucket B: older content that has not been updated recently
stale_mask = (
    (df["content_age_days"] >= 180)
    & (df["days_since_last_update"] >= 180)
)

signal_summary = pd.DataFrame([
    {
        "signal_bucket": "Impression decline",
        "n": int(decline_mask.sum()),
        "verdict": "MIXED",
        "explanation": (
            "The signal identifies observed declines, but seasonality, "
            "demand changes, and measurement issues may also explain them."
        ),
    },
    {
        "signal_bucket": "Stale content",
        "n": int(stale_mask.sum()),
        "verdict": "MIXED",
        "explanation": (
            "Age supports a review candidate, but older content is not "
            "necessarily outdated or in need of a refresh."
        ),
    },
])

display(signal_summary)


,signal_bucket,n,verdict,explanation
0,Impression decline,11101,MIXED,"The signal identifies observed declines, but s..."
1,Stale content,174,MIXED,"Age supports a review candidate, but older con..."


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.